# 05 — Single-Qubit Workflows via rqm-qiskit

`rqm-qiskit` is the **execution bridge** between `rqm-core` math objects and Qiskit circuits.

Instead of building Qiskit circuits manually gate by gate, you describe rotations in terms of quaternions or SU(2) matrices, and `rqm-qiskit` translates them into the appropriate Qiskit gates.

---

## The bridge pattern

```
rqm-core  →  rqm-qiskit  →  Qiskit  →  Simulator / IBM
 (math)       (bridge)     (circuit)    (execution)
```

```python
from rqm_core import SU2
from rqm_qiskit import to_circuit

rotation = SU2.rotation(np.pi / 2, axis=[0, 0, 1])
circuit  = to_circuit(rotation)
```


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from helpers.notebook_style import setup_notebook
from helpers.plotting import draw_bloch_sphere, plot_counts
setup_notebook()

# Try to import rqm-qiskit; fall back to vanilla Qiskit stubs
try:
    import rqm_qiskit
    from rqm_qiskit import to_circuit, run_simulator
    USING_RQM_QISKIT = True
    print(f"rqm-qiskit {rqm_qiskit.__version__} loaded ✓")
except ImportError:
    USING_RQM_QISKIT = False
    print("rqm-qiskit not installed — running vanilla Qiskit demo.")
    print("Install with: pip install rqm-qiskit")


## A clean single-qubit workflow

The goal is to start in $|0\rangle$, apply a rotation, and measure.

With `rqm-qiskit` the workflow is:
1. Define the rotation using `rqm-core`
2. Convert to a Qiskit circuit with `rqm_qiskit.to_circuit()`
3. Run on the Aer simulator
4. Plot the results


In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

def run_circuit(qc, shots=1024):
    """Run a circuit on the local Aer simulator."""
    sim = AerSimulator()
    job = sim.run(qc.decompose(), shots=shots)
    return job.result().get_counts()

if USING_RQM_QISKIT:
    # --- rqm-qiskit path (canonical) ---
    from rqm_core import SU2
    rotation = SU2.rotation(np.pi / 2, axis=[0, 1, 0])   # Ry(π/2)
    qc = to_circuit(rotation)
    qc.measure_all()
    counts = run_simulator(qc, shots=1024)
else:
    # --- vanilla Qiskit fallback ---
    qc = QuantumCircuit(1)
    qc.ry(np.pi / 2, 0)   # same as SU2 Ry(π/2)
    qc.measure_all()
    counts = run_circuit(qc)

print("Measurement counts:", counts)

fig, ax = plt.subplots(figsize=(4, 3))
plot_counts(counts, title=r"$R_y(\pi/2)|0angle$ measurement", ax=ax)
plt.tight_layout()
plt.show()


## Inspecting the circuit

Qiskit lets us draw the circuit to verify it matches our intended rotation.


In [ ]:
print(qc.draw(output="text"))


## Expected results

$R_y(\pi/2)|0\rangle = \cos(\pi/4)|0\rangle + \sin(\pi/4)|1\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle) = |+\rangle$

So we expect roughly 50/50 counts for $|0\rangle$ and $|1\rangle$.

The small deviations from exactly 50% are due to statistical shot noise — not a circuit error.


In [ ]:
# Verify: Ry(π) should give |1> with certainty
if USING_RQM_QISKIT:
    from rqm_core import SU2
    flip = SU2.rotation(np.pi, axis=[0, 1, 0])   # X via Ry(π)
    qc_flip = to_circuit(flip)
    qc_flip.measure_all()
    counts_flip = run_simulator(qc_flip, shots=512)
else:
    qc_flip = QuantumCircuit(1)
    qc_flip.ry(np.pi, 0)
    qc_flip.measure_all()
    counts_flip = run_circuit(qc_flip, shots=512)

print("Ry(π)|0> counts (should be ~all '1'):", counts_flip)

fig, ax = plt.subplots(figsize=(4, 3))
plot_counts(counts_flip, title=r"$R_y(\pi)|0angle$ — should give $|1angle$", ax=ax)
plt.tight_layout()
plt.show()


## Summary

- `rqm-qiskit` bridges `rqm-core` math objects to Qiskit circuits
- The workflow: define rotation → convert to circuit → simulate → plot
- All math stays in `rqm-core`; all execution goes through `rqm-qiskit`
- The vanilla Qiskit fallback confirms the circuit is correct

**Next:** [06_simulator_measurements.ipynb](06_simulator_measurements.ipynb)
